In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="04-youtube-video-search",
    notebook_name="04_interview_walkthrough.ipynb"
)

# YouTube Video Search: Complete Interview Walkthrough

## What This Notebook Is

This is a **complete mock interview simulation** for the YouTube Video Search system design problem. It is written as a dialogue between an interviewer and a staff-level candidate, with timing guidance, scoring rubrics, and the exact phrases that impress interviewers.

Think of this as a **play rehearsal script**. You are the candidate. Read through the dialogue, internalize the structure, and practice saying the answers out loud.

---

## Interview Structure (45 Minutes)

| Phase | Time | What Happens |
|-------|------|-------------|
| 1. Clarify Requirements | 3-5 min | Ask questions, define scope |
| 2. Frame as ML Problem | 3-5 min | Define objective, I/O, ML category |
| 3. Data & Features | 5-7 min | Data sources, feature engineering |
| 4. Model Architecture | 8-10 min | Encoders, training, loss function |
| 5. Evaluation Metrics | 3-5 min | Offline (MRR) and online (watch time) |
| 6. Serving Architecture | 8-10 min | Candidate gen, ranking, re-ranking |
| 7. Follow-ups | 5-8 min | Deep dives, tradeoffs, what-ifs |

## Phase 1: Clarify Requirements (3-5 minutes)

### The Dialogue

---

**Interviewer**: "Design a video search system for YouTube."

**Candidate**: "Great -- I would love to design this. Before I dive in, let me ask a few clarifying questions to make sure I scope this correctly."

> **Why this matters**: Staff engineers ALWAYS clarify before building. Junior engineers jump straight to solutions.

**Candidate**: "First, is the input text-only, or can users search with images or video clips?"

**Interviewer**: "Text queries only."

**Candidate**: "Got it. And the output is a ranked list of videos sorted by relevance to the query?"

**Interviewer**: "Yes."

**Candidate**: "For relevance, should I consider just the video's visual content, or also textual metadata like titles, descriptions, and tags?"

**Interviewer**: "Both visual content and textual metadata."

**Candidate**: "Do we have labeled training data? Something like annotated (query, relevant video) pairs?"

**Interviewer**: "Yes, assume you have 10 million annotated pairs."

**Candidate**: "How many videos are on the platform?"

**Interviewer**: "About 1 billion."

**Candidate**: "And for latency -- is sub-200ms acceptable?"

**Interviewer**: "Yes."

**Candidate**: "Last question -- should results be personalized per user, or should the same query return the same results for everyone?"

**Interviewer**: "No personalization for now. Focus on query relevance."

---

### Key Phrase to Memorize

> "Let me summarize: we are designing a text-to-video search system that returns a ranked list of relevant videos from a corpus of 1 billion, using both visual content and metadata, with 10M labeled pairs for training, no personalization, and sub-200ms latency."

In [ ]:
# === Interview Timing Tracker ===

import matplotlib.pyplot as plt
import numpy as np

phases = [
    'Clarify\nRequirements', 'Frame as\nML Problem', 'Data &\nFeatures',
    'Model\nArchitecture', 'Evaluation\nMetrics', 'Serving\nArchitecture', 'Follow-ups'
]
times = [4, 4, 6, 9, 4, 9, 7]  # minutes
cumulative = np.cumsum([0] + times[:-1])
colors = ['#FFE0B2', '#B3E5FC', '#C8E6C9', '#F8BBD0', '#D1C4E9', '#FFCCBC', '#FFF9C4']

fig, ax = plt.subplots(figsize=(14, 3))

for i, (phase, t, start, color) in enumerate(zip(phases, times, cumulative, colors)):
    ax.barh(0, t, left=start, height=0.6, color=color, edgecolor='black', linewidth=1.5)
    ax.text(start + t/2, 0, f'{phase}\n({t} min)', ha='center', va='center', fontsize=8, fontweight='bold')

ax.set_xlim(0, 45)
ax.set_ylim(-0.5, 0.5)
ax.set_xlabel('Time (minutes)', fontsize=12)
ax.set_yticks([])
ax.set_title('45-Minute Interview Timeline', fontsize=14, fontweight='bold')

# Add minute markers
for m in range(0, 46, 5):
    ax.axvline(x=m, color='gray', linestyle=':', alpha=0.5)
    ax.text(m, -0.4, f'{m}', ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

print('Timing tips:')
print('  - Do NOT spend more than 5 minutes on requirements')
print('  - Model Architecture and Serving Architecture get the MOST time')
print('  - Leave 5-7 minutes for follow-ups (shows depth)')

## Phase 2: Frame as ML Problem (3-5 minutes)

### The Dialogue

---

**Candidate**: "Now let me frame this as an ML problem. The ML objective is to rank videos by their relevance to the text query.

I would frame this as a **representation learning plus ranking** problem. The core idea is to learn a shared embedding space where both text queries and videos are represented as vectors. Relevant query-video pairs should be close together in this space.

Specifically:
- **Input**: A text query (e.g., 'dogs playing indoors')
- **Output**: A ranked list of videos sorted by relevance
- **Approach**: Train a text encoder and a video encoder to map queries and videos into the same embedding space using contrastive learning. At serving time, compute the query embedding and find nearest video embeddings."

**Interviewer**: "Why not just use text matching on video titles?"

**Candidate**: "Great question. Text matching alone misses videos with poor metadata. For example, a video titled 'VID_20240301.mp4' that actually shows dogs playing would be invisible to text search. By using both a visual search path (embedding-based) and a text search path (inverted index), we get the best of both worlds. I call this a **dual-path architecture**."

---

### Key Phrase to Memorize

> "I frame this as representation learning with a dual-path architecture. The visual path uses contrastive learning to map queries and videos into a shared embedding space. The text path uses an inverted index on metadata. A fusing layer combines both paths."

In [ ]:
# === Quick Architecture Sketch (what you'd draw on a whiteboard) ===

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

import matplotlib.patches as mpatches

def box(ax, x, y, w, h, text, color, fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='#333', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize, fontweight='bold')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Query
box(ax, 5, 9, 4, 0.7, 'Text Query', '#FFF3E0', 11)

# Two paths
box(ax, 0.5, 7, 4.5, 1.2, 'VISUAL PATH\nBERT -> Query Emb -> ANN\n(1B video embeddings)', '#E3F2FD', 9)
box(ax, 9, 7, 4.5, 1.2, 'TEXT PATH\nElasticsearch\n(titles, tags, descriptions)', '#E8F5E9', 9)
arrow(ax, 7, 9, 2.75, 8.2)
arrow(ax, 7, 9, 11.25, 8.2)

# Fusing
box(ax, 4, 5, 6, 0.8, 'FUSING LAYER (weighted combination)', '#F3E5F5', 10)
arrow(ax, 2.75, 7, 7, 5.8)
arrow(ax, 11.25, 7, 7, 5.8)

# Ranking
box(ax, 4, 3.5, 6, 0.8, 'RANKING (LTR Model)', '#FFCCBC', 10)
arrow(ax, 7, 5, 7, 4.3)

# Re-ranking
box(ax, 4, 2, 6, 0.8, 'RE-RANKING (business rules)', '#E0F2F1', 10)
arrow(ax, 7, 3.5, 7, 2.8)

# Results
box(ax, 5, 0.5, 4, 0.7, 'Ranked Results', '#FFF3E0', 11)
arrow(ax, 7, 2, 7, 1.2)

# Annotations
ax.text(0.5, 6.5, '~500 candidates', fontsize=9, color='#1565C0', fontstyle='italic')
ax.text(9, 6.5, '~500 candidates', fontsize=9, color='#2E7D32', fontstyle='italic')
ax.text(10.5, 5.2, '~1000 candidates', fontsize=9, color='#7B1FA2', fontstyle='italic')
ax.text(10.5, 3.7, '~100 candidates', fontsize=9, color='#BF360C', fontstyle='italic')
ax.text(10.5, 2.2, '~10-20 results', fontsize=9, color='#00695C', fontstyle='italic')

ax.set_title('Whiteboard Sketch: YouTube Video Search Architecture', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('This is what you draw on the whiteboard in the first 2 minutes.')
print('Then you walk through each component in detail.')

## Phase 3: Data and Features (5-7 minutes)

### The Dialogue

---

**Candidate**: "For training data, we have 10 million (query, video) pairs. These come from click logs -- when a user searches 'cats jumping' and clicks on a video, that becomes a positive pair. Watch time gives us a quality signal: if they watched most of the video, it is a strong positive.

For features, we need to process two types of input:

**Text features** (for queries and video metadata):
- Normalize: lowercase, remove punctuation, NFKD normalization
- Tokenize: WordPiece subword tokenization (BERT-style)
- This handles unseen words gracefully -- 'unforgettable' becomes ['un', '##for', '##get', '##table']

**Video features**:
- Sample 8 frames uniformly from the video (not all 9000 frames -- that would be 100x slower with minimal quality gain)
- Resize to 224x224 and normalize
- Feed through ViT to get frame embeddings
- Average pool frame embeddings into a single video embedding"

**Interviewer**: "Why 8 frames? Why not more?"

**Candidate**: "Great tradeoff question. Adjacent frames are nearly identical, so 8 uniformly sampled frames capture most of the visual diversity. Going from 8 to 16 frames roughly doubles compute but only improves MRR by about 1-2% in practice. For search relevance -- as opposed to action recognition -- the marginal gain from more frames is minimal. I would start with 8 and increase if offline metrics show frame sampling is the bottleneck."

---

### Key Phrase to Memorize

> "We sample 8 frames per video because adjacent frames are nearly identical. This gives us 100x speedup over processing all frames with minimal quality loss. We use ViT for frame encoding and average pooling for temporal aggregation."

## Phase 4: Model Architecture (8-10 minutes)

### The Dialogue

---

**Candidate**: "The core model is a **dual-encoder** architecture trained with contrastive learning.

**Text Encoder** (BERT-based):
- Takes tokenized query, produces a 256-dim L2-normalized embedding
- Pre-trained on large corpus, fine-tuned on our search data

**Video Encoder** (ViT + temporal attention):
- Encodes each sampled frame with ViT
- Temporal attention layer lets the model weight important frames higher
- Projects to 256-dim L2-normalized embedding in the SAME space as text embeddings

**Training**: Symmetric contrastive loss (InfoNCE / CLIP-style).
In each batch of N (query, video) pairs:
- The diagonal of the N x N similarity matrix = positive pairs
- All off-diagonal = negatives
- Loss pushes positives together, negatives apart
- Temperature parameter controls sharpness

Larger batch sizes are critical because they provide more negatives per positive."

**Interviewer**: "Why BERT for the text encoder instead of something lighter?"

**Candidate**: "BERT gives us context-aware embeddings -- 'bank' in 'river bank' gets a different embedding than 'bank account'. For search queries that are often ambiguous, this contextual understanding is crucial. However, if latency is a concern at serving time, we could distill BERT into a smaller model (e.g., DistilBERT or TinyBERT) that is 2-3x faster with ~95% of the quality."

**Interviewer**: "Why contrastive learning instead of a pointwise classifier?"

**Candidate**: "Contrastive learning naturally maps text and video into the same embedding space, which is exactly what we need for ANN retrieval. A pointwise classifier would give us a relevance score but would NOT give us embeddings suitable for efficient nearest-neighbor search. With contrastive learning, we get both: a similarity function AND embeddings that can be indexed."

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# === The Model You'd Describe on a Whiteboard ===

class InterviewDualEncoder(nn.Module):
    """
    The exact model architecture you'd describe in an interview.
    
    Staff-level talking points:
    1. Shared embedding space (text and video in same 256-D space)
    2. L2 normalization (cosine sim = dot product)
    3. Learnable temperature (controls softmax sharpness)
    4. Symmetric loss (text->video AND video->text)
    """
    def __init__(self, text_vocab=30000, embed_dim=256):
        super().__init__()
        # Text encoder (simplified BERT)
        self.text_emb = nn.Embedding(text_vocab, 128)
        self.text_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=512,
                                       batch_first=True),
            num_layers=2
        )
        self.text_proj = nn.Linear(128, embed_dim)
        
        # Video encoder (simplified ViT + temporal pooling)
        self.frame_encoder = nn.Sequential(
            nn.Linear(512, 256),  # Pre-extracted frame features
            nn.ReLU(),
            nn.Linear(256, embed_dim)
        )
        self.temporal_attn = nn.MultiheadAttention(embed_dim, num_heads=4, batch_first=True)
        self.video_proj = nn.Linear(embed_dim, embed_dim)
        
        # Learnable temperature
        self.log_temp = nn.Parameter(torch.log(torch.tensor(0.07)))
    
    def encode_text(self, token_ids):
        x = self.text_emb(token_ids)
        x = self.text_encoder(x)
        x = x.mean(dim=1)  # Mean pooling
        x = self.text_proj(x)
        return F.normalize(x, p=2, dim=1)
    
    def encode_video(self, frame_features):
        # frame_features: (batch, n_frames, 512)
        x = self.frame_encoder(frame_features)  # (batch, n_frames, embed_dim)
        x, _ = self.temporal_attn(x, x, x)  # Temporal self-attention
        x = x.mean(dim=1)  # Average pool across frames
        x = self.video_proj(x)
        return F.normalize(x, p=2, dim=1)
    
    def forward(self, token_ids, frame_features):
        text_emb = self.encode_text(token_ids)
        video_emb = self.encode_video(frame_features)
        
        # Symmetric contrastive loss
        temp = self.log_temp.exp()
        logits = torch.matmul(text_emb, video_emb.T) / temp
        labels = torch.arange(len(text_emb))
        loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
        return loss, text_emb, video_emb


# Quick demo
model = InterviewDualEncoder()
tokens = torch.randint(1, 30000, (4, 10))
frames = torch.randn(4, 8, 512)

loss, text_emb, video_emb = model(tokens, frames)

print('=== Dual Encoder Summary ===')
print(f'  Text embedding shape: {text_emb.shape}')
print(f'  Video embedding shape: {video_emb.shape}')
print(f'  Temperature: {model.log_temp.exp().item():.4f}')
print(f'  Loss: {loss.item():.4f}')
print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'\nKey interview points:')
print(f'  - Both encoders output 256-D L2-normalized vectors')
print(f'  - Similarity = dot product (equals cosine similarity when normalized)')
print(f'  - Symmetric loss: query->video AND video->query')
print(f'  - Learnable temperature controls softmax sharpness')

## Phase 5: Evaluation Metrics (3-5 minutes)

### The Dialogue

---

**Candidate**: "For evaluation, I would use both offline and online metrics.

**Offline metric: MRR (Mean Reciprocal Rank)**.
- For each query in our test set, we know the relevant video
- MRR = average of 1/rank of the first relevant video
- If the relevant video is always ranked 1st, MRR = 1.0 (perfect)
- I choose MRR over Precision@K because with only one relevant video per query, Precision@10 would max at 0.1, which is misleading

**Online metric: Total watch time**.
- Not CTR -- clickbait gets high CTR but low satisfaction
- Not completion rate -- favors short videos unfairly
- Total watch time is the best proxy: users watch more when results are genuinely relevant

**Guardrail metrics**: search abandonment rate, time to first click, user satisfaction surveys."

**Interviewer**: "What about NDCG?"

**Candidate**: "NDCG is excellent when we have graded relevance labels (0-5 scale). If we have human annotators rating query-video pairs on a graded scale, NDCG is strictly better than MRR because it considers both ranking position AND relevance degree. MRR is simpler and works well with binary relevance. I would use MRR as the quick metric and NDCG as the gold standard when graded labels are available."

---

### Key Phrase to Memorize

> "I use MRR offline because it directly measures how quickly we find the relevant video. Online, I use total watch time because it avoids the clickbait problem of CTR and the short-video bias of completion rate."

In [ ]:
# === Metric Comparison: Why MRR? ===

# Demonstrate with concrete examples
scenarios = {
    'Perfect model': [1, 1, 1, 1, 1],
    'Good model': [1, 2, 1, 3, 2],
    'Mediocre model': [3, 5, 2, 8, 4],
    'Bad model': [10, 25, 15, 50, 30],
}

print('=== Why MRR is the Right Metric ===')
print(f'\n{"Model":<18} {"Ranks":<25} {"MRR":>6} {"Precision@5":>13} {"Recall@5":>10}')
print('-' * 75)

for name, ranks in scenarios.items():
    mrr = np.mean([1/r for r in ranks])
    p5 = np.mean([1 if r <= 5 else 0 for r in ranks]) / 5  # precision@5 with 1 relevant
    r5 = np.mean([1 if r <= 5 else 0 for r in ranks])
    print(f'  {name:<16} {str(ranks):<25} {mrr:>6.3f} {p5:>13.3f} {r5:>10.3f}')

print(f'\nNotice:')
print(f'  - Precision@5 maxes at 0.2 (1 relevant / 5 shown) -- misleading!')
print(f'  - Recall@5 is binary (found or not) -- no granularity')
print(f'  - MRR captures HOW QUICKLY we find the relevant video')

## Phase 6: Serving Architecture (8-10 minutes)

### The Dialogue

---

**Candidate**: "The serving system has three main pipelines.

**Pipeline 1: Video Indexing (Offline, when videos are uploaded)**
- New video uploaded -> video encoder produces embedding -> stored in ANN index (FAISS/ScaNN)
- Title, description, tags -> Elasticsearch inverted index
- Auto-tagger model generates tags for videos with poor metadata
- Runs asynchronously, not in the latency-critical path

**Pipeline 2: Candidate Generation (Online, ~10ms)**
- Query -> text encoder -> query embedding
- ANN search over 1B video embeddings -> ~500 visual candidates
- Elasticsearch search on metadata -> ~500 text candidates
- Both paths run IN PARALLEL (async), then merge with weighted scoring

**Pipeline 3: Ranking + Re-ranking (Online, ~55ms)**
- Feature store provides offline + online features for each candidate
- LTR model (neural ranker) scores ~1000 candidates -> top 100
- Re-ranking service applies business rules: diversity (max 2 per channel), freshness boost, content policy, near-duplicate removal
- Final ~10-20 results returned to user

Total latency: ~65ms well within the 200ms budget."

**Interviewer**: "How do you handle a video that was just uploaded?"

**Candidate**: "When a new video is uploaded, the indexing pipeline kicks in asynchronously. The video encoder computes the embedding, and it gets added to the ANN index. The auto-tagger also generates tags for the ES index. This typically takes a few seconds. During that brief window, the video is not searchable -- but that is acceptable since users do not expect instant searchability. If the title and tags are available immediately, the ES path can find it even before the embedding is computed."

---

### Key Phrase to Memorize

> "The key insight is that video embeddings are computed OFFLINE when uploaded, but query embeddings are computed ONLINE at search time. This asymmetry is critical for latency -- we cannot encode a video in real-time, but encoding a short text query takes only 5ms."

In [ ]:
# === Serving Pipeline: End-to-End Simulation ===

import time

class YouTubeSearchPipeline:
    """
    Complete search pipeline simulation with timing.
    This is what you'd walk through on the whiteboard.
    """
    def __init__(self):
        self.timings = {}
    
    def search(self, query):
        total_start = time.time()
        
        # Step 1: Query Understanding (spelling, intent)
        start = time.time()
        processed_query = query.lower().strip()  # Simplified
        self.timings['query_understanding'] = (time.time() - start) * 1000
        
        # Step 2: Query Encoding (text encoder)
        start = time.time()
        query_emb = np.random.randn(256)  # Simulated
        query_emb /= np.linalg.norm(query_emb)
        self.timings['query_encoding'] = (time.time() - start) * 1000
        
        # Step 3: Candidate Generation (both paths in parallel)
        start = time.time()
        visual_candidates = list(range(500))  # Simulated ANN results
        text_candidates = list(range(400, 900))  # Simulated ES results
        all_candidates = list(set(visual_candidates + text_candidates))
        self.timings['candidate_generation'] = (time.time() - start) * 1000
        
        # Step 4: Feature Extraction
        start = time.time()
        features = np.random.randn(len(all_candidates), 8)  # Simulated
        self.timings['feature_extraction'] = (time.time() - start) * 1000
        
        # Step 5: Ranking (LTR model)
        start = time.time()
        scores = np.random.randn(len(all_candidates))  # Simulated model scoring
        top_100_idx = np.argsort(-scores)[:100]
        self.timings['ranking'] = (time.time() - start) * 1000
        
        # Step 6: Re-Ranking (business rules)
        start = time.time()
        final_results = top_100_idx[:10]  # Simulated re-ranking
        self.timings['reranking'] = (time.time() - start) * 1000
        
        self.timings['total'] = (time.time() - total_start) * 1000
        
        return final_results


pipeline = YouTubeSearchPipeline()
results = pipeline.search('funny cat videos')

print('=== End-to-End Pipeline Timing ===')
print(f'\nQuery: "funny cat videos"')
print(f'Results: {len(results)} videos returned')
print(f'\n{"Stage":<25} {"Latency":>10}')
print('-' * 40)

# Use realistic simulated timings
realistic_timings = {
    'Query Understanding': '5 ms',
    'Query Encoding (BERT)': '5 ms',
    'Candidate Gen (parallel)': '12 ms',
    '  - ANN (FAISS)': '10 ms',
    '  - Elasticsearch': '12 ms',
    'Feature Extraction': '8 ms',
    'Ranking (LTR)': '25 ms',
    'Re-Ranking': '3 ms',
    'Serialization': '2 ms',
}

for stage, latency in realistic_timings.items():
    print(f'  {stage:<25} {latency:>8}')

print(f'\n  {"TOTAL":<25} {"~60 ms":>8}')
print(f'  {"Budget":<25} {"200 ms":>8}')
print(f'  {"Headroom":<25} {"~140 ms":>8}')

## Phase 7: Follow-Up Questions (5-8 minutes)

### Common Follow-Ups and Strong Answers

---

**Q1: "How would you handle multi-language support?"**

> "I would use a multilingual text encoder like mBERT or XLM-R that supports 100+ languages. This maps queries in any language into the same embedding space. The video encoder does not change since visual content is language-agnostic. For text search, I would use language-specific analyzers in Elasticsearch."

---

**Q2: "What about position bias in the training data?"**

> "Users click position 1 more than position 5 regardless of true relevance. To address this, I would use inverse propensity weighting -- weight each click by 1/P(click|position). This debiases the training signal. Alternatively, I could train the ranking model with position as a feature, then set position to 0 at serving time."

---

**Q3: "How would you handle head vs. tail queries?"**

> "Head queries like 'music video' are extremely common. I would cache their results and update every few hours. Tail queries like 'how to fix a leaky faucet in a 1970s ranch house' require deeper semantic understanding. For these, the visual search path is more important because the exact terms may not appear in any video title."

---

**Q4: "How do you ensure freshness for news queries?"**

> "Query intent classification is key. If the query is classified as 'news' or 'trending', I would boost the freshness weight in the re-ranking layer. For queries like 'election results', a video from yesterday is much more relevant than one from last year. The re-ranking service applies a time-decay multiplier."

---

**Q5: "What if you had more time -- what would you improve?"**

> "Three things, in order of impact:
> 1. **Audio features** -- process the audio track with an audio encoder. Queries like 'acoustic guitar' need audio understanding.
> 2. **Cross-attention fusion** instead of late fusion -- lets modalities inform each other.
> 3. **Personalization** -- incorporate user watch history as a feature in the ranking model. Same query, different users, different optimal results."

In [ ]:
# === Interview Scoring Rubric ===

rubric = {
    'Requirements Gathering': {
        'Junior': 'Jumps to solution, misses key constraints',
        'Mid': 'Asks some questions, covers basics',
        'Senior': 'Systematic questions, identifies edge cases',
        'Staff': 'Asks about tradeoffs, probes business constraints, summarizes clearly',
    },
    'Problem Framing': {
        'Junior': 'Picks an approach without justification',
        'Mid': 'Frames correctly but shallow reasoning',
        'Senior': 'Explains why dual-path, discusses alternatives',
        'Staff': 'Connects ML objective to business goal, explains tradeoffs between approaches',
    },
    'Model Architecture': {
        'Junior': 'Uses a generic model without customization',
        'Mid': 'Correct architecture with some details',
        'Senior': 'Full architecture with training details',
        'Staff': 'Discusses contrastive vs alternatives, batch size impact, temperature, scaling',
    },
    'Evaluation': {
        'Junior': 'Mentions accuracy',
        'Mid': 'Uses precision/recall',
        'Senior': 'MRR + online metrics',
        'Staff': 'Explains WHY MRR > Precision@K, discusses guardrail metrics and novelty effects',
    },
    'Serving': {
        'Junior': 'Skips serving details',
        'Mid': 'Mentions ANN search',
        'Senior': 'Multi-stage funnel, latency budget',
        'Staff': 'Feature store, async parallel retrieval, quantization, A/B testing framework',
    },
}

print('=== Interview Scoring Rubric ===')
print('\nWhat separates Junior from Staff answers:\n')

for area, levels in rubric.items():
    print(f'--- {area} ---')
    for level, description in levels.items():
        marker = '>>>' if level == 'Staff' else '   '
        print(f'  {marker} {level}: {description}')
    print()

## The 30-Second Pitch (Memorize This)

If asked to summarize your design in 30 seconds:

> "I designed a dual-path video search system. The **visual path** uses contrastive learning to train a BERT text encoder and ViT video encoder in a shared embedding space, with ANN retrieval over 1 billion pre-indexed video embeddings. The **text path** uses Elasticsearch on video metadata with auto-generated tags. A fusing layer combines both paths with weighted scoring. Candidates pass through a **neural LTR model** for ranking, then a **re-ranking service** applies business rules like diversity and freshness. For evaluation, I use **MRR offline** and **total watch time online**. The system serves results in **~65ms** -- well within the 200ms budget."

---

## Key Design Decisions to Defend

| Decision | Defense |
|----------|----------|
| Dual-path (visual + text) | Catches videos with bad metadata; catches clickbait |
| Frame-level over video-level | 100x faster, minimal quality loss for search |
| Contrastive learning | Maps text + video to same space; enables ANN retrieval |
| MRR over Precision@K | With 1 relevant video, P@K is misleadingly low |
| Weighted fusion over ML fusion | Simpler, faster, no additional training needed |
| Auto-tagging | Many videos have poor metadata; improves text search recall |
| Multi-stage funnel | Allows cheap broad retrieval + expensive precise ranking |

## Final Checklist Before the Interview

Go through this checklist to make sure you can speak confidently about each topic:

- [ ] Can I draw the dual-path architecture from memory?
- [ ] Can I explain contrastive loss (InfoNCE) intuitively and mathematically?
- [ ] Do I know why MRR > Precision@K for this problem?
- [ ] Can I walk through the multi-stage funnel (1B -> 1K -> 100 -> 10)?
- [ ] Can I explain three LTR approaches and when to use each?
- [ ] Do I know the latency budget breakdown?
- [ ] Can I explain the feature store (online vs offline features)?
- [ ] Do I know how to handle head vs tail queries?
- [ ] Can I explain position bias and how to address it?
- [ ] Can I give the 30-second pitch without hesitation?

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)